# DATA 245 – Stockout-Aware Forecasting for Fresh Retail
## Dataset Preparation and Baseline Forecasting

This project asks whether stockouts can make strong fresh-retail products appear unpopular. My responsibility is data validation and preparation, a reproducible pilot, chronological splits, simple forecasting baselines, and evaluation. Using FreshRetailNet-50K, this notebook prepares the pilot and benchmarks persistence and a seven-day recent average before the group runs its regression and tree models.

## 1. Imports and Configuration

Paths are rooted at the repository, and the pilot design is centralized here for reproducibility.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'data' / 'freshretail.csv').exists():
    ROOT = Path.cwd().parent
DATA_PATH = ROOT / 'data' / 'freshretail.csv'
PROCESSED_DIR = ROOT / 'data' / 'processed'
RESULTS_DIR = ROOT / 'results'
RANDOM_SEED = 245
N_PILOT_SERIES = 200
EXPECTED_DAYS = 90
SERIES_KEYS = ['store_id', 'product_id']
RECORD_KEYS = [*SERIES_KEYS, 'dt']
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load the Dataset

The source CSV is parsed with `dt` as a date. The preview and schema summary keep the large dataset readable.

In [2]:
df = pd.read_csv(DATA_PATH, parse_dates=['dt'])
required = {'store_id', 'product_id', 'dt', 'sale_amount', 'stock_hour6_22_cnt'}
missing = required.difference(df.columns)
assert not missing, f'Missing required columns: {sorted(missing)}'
assert df['dt'].notna().all()
display(df.head())
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Date range: {df.dt.min().date()} through {df.dt.max().date()}')
print('Columns:', ', '.join(df.columns))

,city_id,store_id,management_group_id,first_category_id,second_category_id,third_category_id,product_id,dt,sale_amount,hours_sale,stock_hour6_22_cnt,hours_stock_status,discount,holiday_flag,activity_flag,precpt,avg_temperature,avg_humidity,avg_wind_level
0,0,0,0,5,6,65,38,2024-03-28,0.1,[0. 0. 0. 0. 0. 0. 0. 0.1 0. 0. 0. 0...,0,[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0],1.0,0,0,1.6999,15.48,73.54,1.97
1,0,0,0,5,6,65,38,2024-03-29,0.1,[0. 0. 0. 0. 0. 0. 0.1 0. 0. 0. 0. 0...,1,[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1],1.0,0,0,3.0190,15.08,76.56,1.71
2,0,0,0,5,6,65,38,2024-03-30,0.0,[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,0,[1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0],1.0,1,0,2.0942,15.91,76.47,1.73
3,0,0,0,5,6,65,38,2024-03-31,0.1,[0. 0. 0. 0. 0. 0. 0. 0. 0.1 0. 0. 0...,11,[0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1],1.0,1,0,1.5618,16.13,77.40,1.76
4,0,0,0,5,6,65,38,2024-04-01,0.2,[0. 0. 0. 0. 0. 0. 0.1 0. 0. 0. 0. 0...,8,[1 1 1 1 1 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1],1.0,0,0,3.5386,15.37,78.26,1.25


Shape: 1,459,198 rows × 19 columns
Date range: 2024-03-28 through 2024-06-25
Columns: city_id, store_id, management_group_id, first_category_id, second_category_id, third_category_id, product_id, dt, sale_amount, hours_sale, stock_hour6_22_cnt, hours_stock_status, discount, holiday_flag, activity_flag, precpt, avg_temperature, avg_humidity, avg_wind_level


## 3. Data Validation and Audit

Audit counts and series coverage are computed from the dataset. Completeness is based on the actual expected date set, not row count alone.

In [3]:
expected_dates = pd.date_range(df.dt.min(), df.dt.max(), freq='D')
assert len(expected_dates) == EXPECTED_DAYS and df.dt.nunique() == EXPECTED_DAYS
null_counts = df.isna().sum()
duplicate_count = int(df.duplicated(RECORD_KEYS).sum())
series_lengths = df.groupby(SERIES_KEYS, observed=True).size().rename('row_count')
series_dates = df.groupby(SERIES_KEYS, observed=True).dt.agg(lambda s: frozenset(s))
expected_date_set = frozenset(expected_dates)
complete_mask = series_dates.eq(expected_date_set)
complete_index = series_dates.index[complete_mask]
incomplete_index = series_dates.index[~complete_mask]
length_distribution = series_lengths.value_counts().sort_index()
audit = pd.DataFrame({'measure':['Rows','Columns','Date range','Distinct dates','Store-product series','Duplicate series-date rows',f'Complete {EXPECTED_DAYS}-date series','Incomplete series'], 'value':[len(df),len(df.columns),f'{df.dt.min().date()} to {df.dt.max().date()}',df.dt.nunique(),len(series_lengths),duplicate_count,int(complete_mask.sum()),int((~complete_mask).sum())]})
display(audit)
print('Series length distribution:'); display(length_distribution.rename_axis('rows_per_series').rename('series_count').to_frame())
print('Null counts by column:'); display(null_counts.rename('null_count').to_frame())
assert null_counts.sum() == 0 and duplicate_count == 0

,measure,value
0,Rows,1459198
1,Columns,19
2,Date range,2024-03-28 to 2024-06-25
3,Distinct dates,90
4,Store-product series,16214
5,Duplicate series-date rows,0
6,Complete 90-date series,16213
7,Incomplete series,1


Series length distribution:


,series_count
rows_per_series,
28,1
90,16213


Null counts by column:


,null_count
city_id,0
store_id,0
management_group_id,0
first_category_id,0
second_category_id,0
third_category_id,0
product_id,0
dt,0
sale_amount,0
hours_sale,0


The current dataset is very clean, with no null values or duplicate store-product-date records. One series has only 28 days, so it should be excluded before lagged features and chronological modeling.

## 4. Chronological Ordering

Sorting within each store-product series ensures that lags and forecasts refer only to earlier observations.

In [4]:
df = df.sort_values(RECORD_KEYS, kind='mergesort').reset_index(drop=True)
chronological = df.groupby(SERIES_KEYS, sort=False, observed=True).dt.apply(lambda s: s.is_monotonic_increasing)
assert chronological.all()
print(f'Chronological order verified for {len(chronological):,} series.')

Chronological order verified for 16,214 series.


## 5. Complete-Series Filtering

Each retained series must contain all 90 expected calendar dates. Incomplete series are saved with their missing-date details.

In [5]:
excluded_rows = []
for (store_id, product_id), dates in series_dates.loc[~complete_mask].items():
    excluded_rows.append({'store_id':store_id, 'product_id':product_id, 'row_count':int(series_lengths.loc[(store_id,product_id)]), 'distinct_dates':len(dates), 'missing_dates':';'.join(d.strftime('%Y-%m-%d') for d in expected_dates if d not in dates), 'unexpected_dates':';'.join(sorted(d.strftime('%Y-%m-%d') for d in dates - expected_date_set))})
excluded = pd.DataFrame(excluded_rows, columns=['store_id','product_id','row_count','distinct_dates','missing_dates','unexpected_dates'])
excluded.to_csv(PROCESSED_DIR / 'excluded_series.csv', index=False)
complete_series = pd.DataFrame(complete_index.tolist(), columns=SERIES_KEYS)
print(f'Complete series: {len(complete_series):,}; incomplete series: {len(excluded):,}')
display(excluded)

Complete series: 16,213; incomplete series: 1


,store_id,product_id,row_count,distinct_dates,missing_dates,unexpected_dates
0,420,764,28,28,2024-04-25;2024-04-26;2024-04-27;2024-04-28;20...,


The incomplete series is excluded from pilot sampling. The separate CSV preserves the exact exclusion details for review.

## 6. Reproducible 200-Series Pilot

Sampling occurs at the store-product level with a fixed seed, so reruns select the same pairs.

In [6]:
assert len(complete_series) >= N_PILOT_SERIES
manifest = complete_series.sample(n=N_PILOT_SERIES, random_state=RANDOM_SEED).sort_values(SERIES_KEYS).reset_index(drop=True)
manifest.to_csv(PROCESSED_DIR / 'pilot_series_manifest.csv', index=False)
pilot = df.merge(manifest, on=SERIES_KEYS, how='inner', validate='many_to_one').sort_values(RECORD_KEYS, kind='mergesort').reset_index(drop=True)
expected_pilot_rows = N_PILOT_SERIES * EXPECTED_DAYS
assert len(pilot) == expected_pilot_rows
print(f'Sampled series: {len(manifest):,}; pilot rows: {len(pilot):,} (expected {expected_pilot_rows:,})')

Sampled series: 200; pilot rows: 18,000 (expected 18,000)


## 7. Day Number and Chronological Splits

Shared calendar dates map to the same day number for every series. Chronological splits preserve temporal order and avoid random train/test shuffling.

In [7]:
date_to_day = {date: day for day, date in enumerate(expected_dates, start=1)}
pilot['day_number'] = pilot.dt.map(date_to_day).astype('int8')
assert pilot.groupby('dt').day_number.nunique().eq(1).all()
pilot['split'] = pd.cut(pilot.day_number, bins=[0,60,75,90], labels=['train','validation','test']).astype('string')
split_summary = pilot.groupby('split', observed=False).agg(start_date=('dt','min'), end_date=('dt','max'), rows=('dt','size')).reset_index()
split_summary['start_date'] = split_summary.start_date.dt.date
split_summary['end_date'] = split_summary.end_date.dt.date
display(split_summary)

,split,start_date,end_date,rows
0,test,2024-06-11,2024-06-25,3000
1,train,2024-03-28,2024-05-26,12000
2,validation,2024-05-27,2024-06-10,3000


## 8. Persistence Baseline

For each store-product series, the persistence forecast for day *t* is the sale amount from day *t ? 1*. The first observation in each series has no valid prediction and is excluded from evaluation.

In [ ]:
sales = pilot.groupby(SERIES_KEYS, sort=False, observed=True)['sale_amount']
pilot['pred_persistence'] = sales.shift(1)


## 9. Seven-Day Recent-Average Baseline

For each series, the forecast for day *t* is the mean sale amount from days *t ? 7* through *t ? 1*. Shifting the series before applying the rolling mean ensures the target day is excluded.

In [8]:
pilot['pred_recent_avg_7'] = sales.transform(
    lambda s: s.shift(1).rolling(window=7, min_periods=7).mean()
)
pilot.to_csv(PROCESSED_DIR / 'pilot_200_series.csv', index=False)
display(pilot[[*SERIES_KEYS, 'dt', 'day_number', 'sale_amount', 'pred_persistence', 'pred_recent_avg_7']].head(10))


,store_id,product_id,dt,day_number,sale_amount,pred_persistence,pred_recent_avg_7
0,1,200,2024-03-28,1,0.4,NaN,NaN
1,1,200,2024-03-29,2,0.0,0.4,NaN
2,1,200,2024-03-30,3,0.5,0.0,NaN
3,1,200,2024-03-31,4,0.6,0.5,NaN
4,1,200,2024-04-01,5,0.4,0.6,NaN
5,1,200,2024-04-02,6,0.1,0.4,NaN
6,1,200,2024-04-03,7,0.3,0.1,NaN
7,1,200,2024-04-04,8,0.4,0.3,0.328571
8,1,200,2024-04-05,9,0.5,0.4,0.328571
9,1,200,2024-04-06,10,0.8,0.5,0.400000


## 10. Evaluation Metrics

Reusable functions calculate MAE, RMSE, and signed bias, defined as `mean(predicted ? actual)`. Positive bias indicates overprediction; negative bias indicates underprediction. Evaluation is limited to validation and test, with unavailable predictions excluded.

In [ ]:
def calculate_metrics(actual: pd.Series, predicted: pd.Series) -> dict[str, float]:
    error = predicted - actual
    return {'MAE':error.abs().mean(), 'RMSE':np.sqrt(error.pow(2).mean()), 'bias':error.mean()}



## 11. Baseline Results

The following table compares both baselines separately on validation and test. Training metrics are not presented as primary results.

In [9]:
metric_rows, prediction_frames = [], []

for model, pred_col in [('persistence','pred_persistence'), ('recent_avg_7','pred_recent_avg_7')]:
    for split in ['validation','test']:
        scored = pilot.loc[pilot.split.eq(split) & pilot[pred_col].notna(), [*SERIES_KEYS,'dt','day_number','split','sale_amount',pred_col]].copy()
        metric_rows.append({'model':model, 'split':split, 'n_observations':len(scored), **calculate_metrics(scored.sale_amount, scored[pred_col])})
        scored = scored.rename(columns={pred_col:'prediction'}); scored.insert(0,'model',model); prediction_frames.append(scored)
baseline_metrics = pd.DataFrame(metric_rows)
baseline_predictions = pd.concat(prediction_frames, ignore_index=True)
baseline_predictions.to_csv(RESULTS_DIR / 'baseline_predictions.csv', index=False)
baseline_metrics.to_csv(RESULTS_DIR / 'baseline_metrics.csv', index=False, float_format='%.6f')
display(baseline_metrics.round(6))

,model,split,n_observations,MAE,RMSE,bias
0,persistence,validation,3000,0.417757,0.677441,0.002357
1,persistence,test,3000,0.444063,0.697380,-0.002777
2,recent_avg_7,validation,3000,0.337380,0.551965,-0.007276
3,recent_avg_7,test,3000,0.362622,0.564407,-0.051647


## 12. Preliminary Interpretation

The seven-day recent-average baseline has lower MAE and RMSE than persistence on both validation and test. Persistence has near-zero test bias; the seven-day average has more negative test bias, indicating some underprediction. These are baseline results only and do not yet answer whether stockout-aware features improve forecasting. The group will compare stockout-aware with sales-only inputs in its later regression and tree models.

## 13. Leakage Prevention and Reproducibility Checks

These implementation checks are not a formal leakage audit. They verify chronological order, prior-only forecasts, the fixed seed, common date splits, and the absence of random row shuffling.

In [10]:
assert chronological.all() and RANDOM_SEED == 245 and len(manifest) == N_PILOT_SERIES
assert pilot.groupby('dt').day_number.nunique().eq(1).all()
assert pilot.groupby(SERIES_KEYS, observed=True).split.nunique().eq(3).all()
assert pilot.pred_persistence.isna().equals(pilot.groupby(SERIES_KEYS, observed=True).cumcount().eq(0))
assert pilot.groupby(SERIES_KEYS, observed=True).pred_recent_avg_7.apply(lambda s: s.iloc[:7].isna().all() and s.iloc[7:].notna().all()).all()
# Recompute an example directly to verify the target day is excluded.
example = pilot.groupby(SERIES_KEYS, sort=False, observed=True).head(10).groupby(SERIES_KEYS, sort=False, observed=True).head(10).iloc[:10]
sales_example = example.sale_amount.to_numpy()
np.testing.assert_allclose(example.pred_persistence.iloc[1:].to_numpy(), sales_example[:-1])
np.testing.assert_allclose(example.pred_recent_avg_7.iloc[7:].to_numpy(), [sales_example[i-7:i].mean() for i in range(7, len(sales_example))])
print('Chronology, prior-only forecasts, fixed-seed sampling, and common splits verified.')

Chronology, prior-only forecasts, fixed-seed sampling, and common splits verified.


## 14. Generated Output Files

The pipeline writes the following reproducible artifacts:

In [11]:
# Keep the audit report in sync with the data computed above.
audit_lines = ['FreshRetail data audit', f'Rows: {len(df):,}', f'Columns: {len(df.columns)}', f'Date range: {df.dt.min().date()} through {df.dt.max().date()}', f'Distinct dates: {df.dt.nunique()}', f'Unique store-product series: {len(series_lengths):,}', f'Duplicate (store_id, product_id, dt) rows: {duplicate_count}', f'Complete {EXPECTED_DAYS}-date series: {int(complete_mask.sum()):,}', f'Incomplete series: {int((~complete_mask).sum()):,}', 'Series-length distribution (rows per series: series count):']
audit_lines += [f'  {int(k)}: {int(v):,}' for k,v in length_distribution.items()]
audit_lines += ['Null count by column:'] + [f'  {col}: {int(value):,}' for col,value in null_counts.items()]
audit_lines += ['Incomplete series details (also saved to data/processed/excluded_series.csv):']
audit_lines += [f'  store={r.store_id}, product={r.product_id}, rows={r.row_count}, distinct_dates={r.distinct_dates}, missing={r.missing_dates or "none"}' for r in excluded.itertuples(index=False)] or ['  None']
(RESULTS_DIR / 'data_audit.txt').write_text('\n'.join(audit_lines) + '\n', encoding='utf-8')
outputs = ['data/processed/excluded_series.csv','data/processed/pilot_series_manifest.csv','data/processed/pilot_200_series.csv','results/data_audit.txt','results/baseline_predictions.csv','results/baseline_metrics.csv']
print('\n'.join(outputs))

data/processed/excluded_series.csv
data/processed/pilot_series_manifest.csv
data/processed/pilot_200_series.csv
results/data_audit.txt
results/baseline_predictions.csv
results/baseline_metrics.csv


## 15. Conclusion

Dataset validation is complete, the reproducible 200-series pilot is prepared, and chronological train/validation/test splits are established. Baseline performance has been measured. These outputs are ready for the group?s later sales-only versus stockout-aware model comparison.